# SOM Node Statistics

By: Ty Janoski

Standalone notebook for per-node precipitation and tropical cyclone analysis. Reads BMU assignments produced by `Z500_and_IVT_SOM_training.ipynb`.

In [26]:
import os
from itertools import combinations

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scienceplots  # noqa: F401
from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu

plt.style.use(["science", "nature", "grid"])
plt.rcParams["text.usetex"] = True

In [27]:
# Set MOISTURE_VAR to "IVT" or "tcwv" (precipitable water)
MOISTURE_VAR = "IVT"

# SOM dimensions (must match the trained SOM)
xdim, ydim = 2, 2
n_nodes = xdim * ydim

_lbl = MOISTURE_VAR.lower()
fig_dir = f"figs/Z500-and-{_lbl}-SOM"
bmu_csv = f"data/som_2x2_bmus_{MOISTURE_VAR}.csv"
precip_path = "precip_data_and_tc_association_code/"
ibtracs_path = "precip_data_and_tc_association_code/ibtracs.NA.list.v04r01.processed_6hrly.statslp3.csv"

os.makedirs(fig_dir, exist_ok=True)

## ASOS Precipitation by SOM Node

In [28]:
# Load hourly precipitation data from four NYC-area ASOS sites
sites = {
    "JFK": np.load(f"{precip_path}jfk7_14_25.npy", allow_pickle=True),
    "LGA": np.load(f"{precip_path}lga7_14_25.npy", allow_pickle=True),
    "Central Park": np.load(f"{precip_path}cp7_14_25.npy", allow_pickle=True),
    "EWR": np.load(f"{precip_path}zewr_14_25.npy", allow_pickle=True),
}
precip_dfs = {}
for name, arr in sites.items():
    df = pd.DataFrame(arr, columns=["precip", "time"])
    df["time"] = pd.to_datetime(df["time"])
    df = df.set_index("time").sort_index()
    df["precip"] = pd.to_numeric(df["precip"], errors="coerce")
    precip_dfs[name] = df

# Load BMU assignments and convert UTC → local time
bmu_df = pd.read_csv(bmu_csv)
bmu_df["timestamp"] = pd.to_datetime(bmu_df["timestamp"])
bmu_df["timestamp_local"] = (
    bmu_df["timestamp"].dt.tz_localize("UTC").dt.tz_convert("EST").dt.tz_localize(None)
)

# For each event find the max hourly rainfall across all sites within ±window_hours
window_hours = 6
max_precip = []
for _, row in bmu_df.iterrows():
    event_time = row["timestamp_local"]
    start = event_time - pd.Timedelta(hours=window_hours)
    end = event_time + pd.Timedelta(hours=window_hours)
    site_maxes = []
    for name, df in precip_dfs.items():
        window_data = df.loc[start:end, "precip"]
        if len(window_data) > 0:
            site_maxes.append(window_data.max())
    valid_maxes = [v for v in site_maxes if not np.isnan(v)]
    max_precip.append(max(valid_maxes) if valid_maxes else 0.0)

bmu_df["max_precip_in"] = max_precip
print(
    f"Matched {len(bmu_df)} events | range: {bmu_df['max_precip_in'].min():.2f}–{bmu_df['max_precip_in'].max():.2f} in"
)


Matched 117 events | range: 0.00–3.62 in


In [29]:
# Per-node precipitation statistics
node_stats = (
    bmu_df.groupby(["node_i", "node_j"])["max_precip_in"]
    .agg(count="count", mean="mean", median="median", std="std")
    .round(2)
)
print(node_stats)


               count  mean  median   std
node_i node_j                           
0      0          25  1.09    1.01  0.49
       1          33  0.85    0.77  0.58
1      0          35  1.06    1.03  0.49
       1          24  0.88    0.78  0.72


In [30]:
# Create histogram of max hourly rainfall for each SOM node
fig, axes = plt.subplots(ydim, xdim, figsize=(6, 4), constrained_layout=True, dpi=600)

# Define consistent bins for all histograms (in inches)
bins = np.arange(0, 3.76, 0.25)

# Colors for each node (matching other plots)
colors = ["steelblue", "darkorange", "seagreen", "firebrick"]

for i in range(xdim):
    for j in range(ydim):
        ax = axes[j, i]

        # Filter events for this node
        node_data = bmu_df[(bmu_df["node_i"] == i) & (bmu_df["node_j"] == j)][
            "max_precip_in"
        ].dropna()

        # Plot histogram
        ax.hist(
            node_data,
            bins=bins,
            color="teal",
            alpha=0.9,
            edgecolor="white",
            linewidth=0.5,
        )

        # Add statistics
        n = len(node_data)
        median = node_data.median() if n > 0 else np.nan
        mean = node_data.mean() if n > 0 else np.nan

        # Add vertical line for median
        if n > 0:
            ax.axvline(
                median,
                color="red",
                linestyle="--",
                linewidth=1,
                label=f'Median: {median:.2f}"',
            )

        ax.set_title(f"({i},{j})  N={n}", fontsize=6)
        ax.set_xlim(0, 3.75)
        ax.set_ylim(0, 12)
        ax.set_yticks(np.arange(0, 13, 2))
        ax.tick_params(axis="both", labelsize=5)
        ax.grid(True, linewidth=0.3, alpha=0.5, axis="y")

        # Only add x-label on bottom row
        if j == ydim - 1:
            ax.set_xlabel("Max Hourly Precip (in)", fontsize=5)

        # Only add y-label on left column
        if i == 0:
            ax.set_ylabel("Count", fontsize=5)

        # Add legend with median
        if n > 0:
            ax.legend(fontsize=4, loc="upper right")

plt.suptitle(
    "Maximum Hourly Rainfall ($\\pm$6 hr window) by SOM Node",
    fontsize=8,
    y=1.02,
)
plt.savefig(
    f"{fig_dir}/Z500_and_{_lbl}_som_max_precip_histograms.png",
    bbox_inches="tight",
)
plt.close()

print(f"Saved histogram to {fig_dir}/Z500_and_{_lbl}_som_max_precip_histograms.png")


Saved histogram to figs/Z500-and-ivt-SOM/Z500_and_ivt_som_max_precip_histograms.png


## Tropical Cyclone Association by SOM Node

Flash flood event times are cross-referenced with IBTrACS to identify events where a tropical system was present within the analysis domain (30–54°N, 100–60°W).

In [31]:
# Load IBTrACS data
ibtracs_path = "precip_data_and_tc_association_code/ibtracs.NA.list.v04r01.processed_6hrly.statslp3.csv"
ibtracs = pd.read_csv(ibtracs_path)
ibtracs["ISO_TIME"] = pd.to_datetime(ibtracs["ISO_TIME"])

# Define domain bounds (same as IVT/Z500 data)
lat_min, lat_max = 30.0, 54.0
lon_min, lon_max = -100.0, -60.0

# Filter IBTrACS to our domain
ibtracs_domain = ibtracs[
    (ibtracs["LAT"] >= lat_min)
    & (ibtracs["LAT"] <= lat_max)
    & (ibtracs["LON"] >= lon_min)
    & (ibtracs["LON"] <= lon_max)
].copy()

print(f"Total IBTrACS records: {len(ibtracs):,}")
print(f"Records within domain: {len(ibtracs_domain):,}")
print(f"Unique storms in domain: {ibtracs_domain['SID'].nunique()}")


Total IBTrACS records: 64,320
Records within domain: 12,035
Unique storms in domain: 1185


In [32]:
# Cross-reference flash flood events with tropical cyclones
# Look for TCs within the domain within ±12 hours of each event

time_window_hours = 6

tc_associations = []
for _, row in bmu_df.iterrows():
    event_time = row["timestamp"]
    node_i, node_j = row["node_i"], row["node_j"]

    # Find TCs within time window
    time_mask = (
        ibtracs_domain["ISO_TIME"] >= event_time - pd.Timedelta(hours=time_window_hours)
    ) & (
        ibtracs_domain["ISO_TIME"] <= event_time + pd.Timedelta(hours=time_window_hours)
    )
    matching_tcs = ibtracs_domain[time_mask]

    if len(matching_tcs) > 0:
        # Get unique storm IDs and names
        storm_ids = matching_tcs["SID"].unique()
        tc_associations.append(
            {
                "timestamp": event_time,
                "node_i": node_i,
                "node_j": node_j,
                "tc_present": True,
                "n_storms": len(storm_ids),
                "storm_ids": ", ".join(storm_ids),
                "storm_status": matching_tcs["STAT"].mode().iloc[0]
                if len(matching_tcs["STAT"].dropna()) > 0
                else "Unknown",
            }
        )
    else:
        tc_associations.append(
            {
                "timestamp": event_time,
                "node_i": node_i,
                "node_j": node_j,
                "tc_present": False,
                "n_storms": 0,
                "storm_ids": "",
                "storm_status": "",
            }
        )

tc_df = pd.DataFrame(tc_associations)

# Summary statistics
n_tc_events = tc_df["tc_present"].sum()
print(
    f"Flash flood events associated with TCs: {n_tc_events}/{len(tc_df)} ({100 * n_tc_events / len(tc_df):.1f}%)"
)
print(f"\nTC-associated events by SOM node:")
for i in range(xdim):
    for j in range(ydim):
        node_data = tc_df[(tc_df["node_i"] == i) & (tc_df["node_j"] == j)]
        tc_count = node_data["tc_present"].sum()
        total = len(node_data)
        pct = 100 * tc_count / total if total > 0 else 0
        print(f"  Node ({i},{j}): {tc_count}/{total} ({pct:.1f}%)")


# List TC-associated flash flood events
tc_events = tc_df[tc_df["tc_present"]].copy()
tc_events = tc_events.sort_values("timestamp")

print("TC-Associated Flash Flood Events:")
print("-" * 80)
for _, row in tc_events.iterrows():
    print(
        f"{row['timestamp'].strftime('%Y-%m-%d %H:%M')} | "
        f"Node ({row['node_i']},{row['node_j']}) | "
        f"Status: {row['storm_status']:>3} | "
        f"{row['storm_ids']}"
    )


Flash flood events associated with TCs: 22/117 (18.8%)

TC-associated events by SOM node:
  Node (0,0): 5/25 (20.0%)
  Node (0,1): 7/33 (21.2%)
  Node (1,0): 7/35 (20.0%)
  Node (1,1): 3/24 (12.5%)
TC-Associated Flash Flood Events:
--------------------------------------------------------------------------------
1996-07-13 13:00 | Node (1,0) | Status:  TS | 1996187N10326
1996-09-08 20:00 | Node (1,1) | Status:  EX | 1996237N14339
1997-07-16 00:00 | Node (0,1) | Status:  TS | 1997194N31286
1999-09-16 16:00 | Node (1,0) | Status:  HU | 1999251N15314
2001-06-17 15:00 | Node (0,0) | Status:  SS | 2001157N28265
2002-09-02 13:00 | Node (0,1) | Status:  TS | 2002245N29281
2004-09-08 10:00 | Node (0,0) | Status:  TD | 2004238N11325
2004-09-18 13:00 | Node (0,1) | Status:  EX | 2004247N10332
2004-09-28 21:00 | Node (0,1) | Status:  EX | 2004258N16300
2005-07-06 23:00 | Node (1,1) | Status:  TD | 2005185N18273
2005-10-14 21:00 | Node (0,1) | Status:  EX | 2005281N26303
2006-07-21 21:00 | Node (0,

In [33]:
# Create bar chart showing TC vs non-TC events by SOM node
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), dpi=600, constrained_layout=True)

# Compute counts per node
tc_counts = np.zeros((xdim, ydim))
non_tc_counts = np.zeros((xdim, ydim))

for i in range(xdim):
    for j in range(ydim):
        node_data = tc_df[(tc_df["node_i"] == i) & (tc_df["node_j"] == j)]
        tc_counts[i, j] = node_data["tc_present"].sum()
        non_tc_counts[i, j] = (~node_data["tc_present"]).sum()

# Left panel: Stacked bar chart
ax = axes[0]
node_labels = [f"({i},{j})" for i in range(xdim) for j in range(ydim)]
x = np.arange(len(node_labels))
width = 0.6

tc_flat = tc_counts.flatten()
non_tc_flat = non_tc_counts.flatten()

bars1 = ax.bar(x, non_tc_flat, width, label="Non-TC", color="steelblue", alpha=0.9)
bars2 = ax.bar(
    x,
    tc_flat,
    width,
    bottom=non_tc_flat,
    label="TC-Associated",
    color="coral",
    alpha=0.9,
)

ax.set_xlabel("SOM Node", fontsize=7)
ax.set_ylabel("Number of Events", fontsize=7)
ax.set_title("Flash Flood Events by TC Association", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(node_labels, fontsize=6)
ax.legend(fontsize=6, loc="upper right")
ax.grid(True, linewidth=0.3, alpha=0.5, axis="y")

# Right panel: TC percentage by node
ax = axes[1]
totals = tc_flat + non_tc_flat
tc_pct = 100 * tc_flat / totals

bars = ax.bar(x, tc_pct, width, color="coral", alpha=0.9, edgecolor="white")

# Add percentage labels on bars
for bar, pct, tc, total in zip(bars, tc_pct, tc_flat, totals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{int(tc)}/{int(total)}",
        ha="center",
        va="bottom",
        fontsize=6,
    )

# Add overall average line
overall_pct = 100 * tc_flat.sum() / totals.sum()
ax.axhline(
    overall_pct,
    color="red",
    linestyle="--",
    linewidth=1,
    label=f"Overall: {overall_pct:.1f}\\%",
)

ax.set_xlabel("SOM Node", fontsize=7)
ax.set_ylabel("TC-Associated Events (\\%)", fontsize=7)
ax.set_title("Percentage of Events with TC Influence", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(node_labels, fontsize=6)
ax.set_ylim(0, max(tc_pct) + 15)
ax.legend(fontsize=6, loc="upper right")
ax.grid(True, linewidth=0.3, alpha=0.5, axis="y")

plt.savefig(f"{fig_dir}/Z500_and_{_lbl}_som_tc_association.png", bbox_inches="tight")
plt.close()

print(
    f"Saved TC association figure to {fig_dir}/Z500_and_{_lbl}_som_tc_association.png"
)


Saved TC association figure to figs/Z500-and-ivt-SOM/Z500_and_ivt_som_tc_association.png


In [34]:
# Statistical test: Is TC association different across SOM nodes?
# Build 2xN contingency table: rows = TC/non-TC, columns = nodes
contingency_tc = np.array([tc_flat, non_tc_flat])

print("=" * 60)
print("CHI-SQUARE TEST: TC Association vs SOM Node")
print("=" * 60)
contingency_df = pd.DataFrame(
    {
        lbl: [int(tc), int(non_tc)]
        for lbl, tc, non_tc in zip(node_labels, tc_flat, non_tc_flat)
    },
    index=["TC", "Non-TC"],
)
contingency_df["Total"] = contingency_df.sum(axis=1)
print("\nContingency Table:")
print(contingency_df)

# Chi-square test
chi2, p_value, dof, expected = chi2_contingency(contingency_tc)

print(f"\nChi-square statistic: {chi2:.3f}")
print(f"Degrees of freedom:   {dof}")
print(f"p-value:              {p_value:.4f}")

# Check expected cell counts
min_expected = expected.min()
print(f"\nMinimum expected count: {min_expected:.2f}")

if min_expected < 5:
    print("⚠ Warning: Some expected counts < 5; interpret with caution")

if p_value < 0.05:
    print("\n→ Result: REJECT H₀ at α=0.05. TC association differs across SOM nodes.")
else:
    print(
        "\n→ Result: FAIL TO REJECT H₀. No significant difference in TC association across nodes."
    )

# Pairwise Fisher's exact tests
print("\n" + "=" * 60)
print("PAIRWISE FISHER'S EXACT TESTS (Bonferroni-corrected)")
print("=" * 60)

pairs = list(combinations(range(n_nodes), 2))
n_comparisons = len(pairs)
alpha_corrected = 0.05 / n_comparisons

print(f"Number of comparisons: {n_comparisons}")
print(f"Bonferroni-corrected α: {alpha_corrected:.4f}\n")

for idx1, idx2 in pairs:
    # 2x2 table for this pair
    table = np.array(
        [[tc_flat[idx1], tc_flat[idx2]], [non_tc_flat[idx1], non_tc_flat[idx2]]]
    )
    odds_ratio, p = fisher_exact(table)
    sig = "***" if p < alpha_corrected else ""
    print(
        f"{node_labels[idx1]} vs {node_labels[idx2]}: OR={odds_ratio:.2f}, p={p:.4f} {sig}"
    )


CHI-SQUARE TEST: TC Association vs SOM Node

Contingency Table:
        (0,0)  (0,1)  (1,0)  (1,1)  Total
TC          5      7      7      3     22
Non-TC     20     26     28     21     95

Chi-square statistic: 0.806
Degrees of freedom:   3
p-value:              0.8480

Minimum expected count: 4.51
⚠ Warning: Some expected counts < 5; interpret with caution

→ Result: FAIL TO REJECT H₀. No significant difference in TC association across nodes.

PAIRWISE FISHER'S EXACT TESTS (Bonferroni-corrected)
Number of comparisons: 6
Bonferroni-corrected α: 0.0083

(0,0) vs (0,1): OR=0.93, p=1.0000 
(0,0) vs (1,0): OR=1.00, p=1.0000 
(0,0) vs (1,1): OR=1.75, p=0.7019 
(0,1) vs (1,0): OR=1.08, p=1.0000 
(0,1) vs (1,1): OR=1.88, p=0.4939 
(1,0) vs (1,1): OR=1.75, p=0.5059 


In [ ]:
# Map showing TC positions during flash flood events, colored by SOM node
fig, ax = plt.subplots(
    figsize=(8, 5),
    subplot_kw={"projection": ccrs.PlateCarree()},
    dpi=600,
)

# Define colors for each node
node_colors = {
    (0, 0): "tab:blue",
    (0, 1): "tab:orange",
    (1, 0): "tab:green",
    (1, 1): "tab:red",
}

# Collect all track coordinates for dynamic extent
all_lons, all_lats = [], []

# For each TC-associated event, plot full track + highlight window
for _, row in tc_events.iterrows():
    event_time = row["timestamp"]
    node_i, node_j = int(row["node_i"]), int(row["node_j"])
    storm_ids = row["storm_ids"].split(", ")
    color = node_colors[(node_i, node_j)]

    for sid in storm_ids:
        full_track = ibtracs[ibtracs["SID"] == sid].copy().sort_values("ISO_TIME")
        if len(full_track) < 2:
            continue

        all_lons.extend(full_track["LON"].tolist())
        all_lats.extend(full_track["LAT"].tolist())

        # Full track: thin + low alpha
        ax.plot(
            full_track["LON"],
            full_track["LAT"],
            color=color,
            linewidth=0.8,
            alpha=0.3,
            transform=ccrs.PlateCarree(),
        )

        # ±48 hr window segment: thick + full alpha
        window_mask = (
            full_track["ISO_TIME"] >= event_time - pd.Timedelta(hours=48)
        ) & (full_track["ISO_TIME"] <= event_time + pd.Timedelta(hours=48))
        window_track = full_track[window_mask]
        if len(window_track) > 1:
            ax.plot(
                window_track["LON"],
                window_track["LAT"],
                color=color,
                linewidth=2.0,
                alpha=0.85,
                transform=ccrs.PlateCarree(),
            )

        # Bracket markers ( | ) at the ±48 hr boundaries
        for delta_hrs in [-48, 48]:
            target = event_time + pd.Timedelta(hours=delta_hrs)
            diffs = (full_track["ISO_TIME"] - target).abs()
            idx = diffs.idxmin()
            if diffs[idx] <= pd.Timedelta(hours=6):
                pt = full_track.loc[idx]
                ax.scatter(
                    pt["LON"],
                    pt["LAT"],
                    color=color,
                    s=60,
                    marker="|",
                    linewidths=1.5,
                    zorder=6,
                    transform=ccrs.PlateCarree(),
                )

        # Dot at event time
        closest_idx = (full_track["ISO_TIME"] - event_time).abs().idxmin()
        closest = full_track.loc[closest_idx]
        ax.scatter(
            closest["LON"],
            closest["LAT"],
            color=color,
            s=40,
            marker="o",
            edgecolor="black",
            linewidth=0.5,
            zorder=7,
            transform=ccrs.PlateCarree(),
        )

# Plot domain box
ax.plot(
    [lon_min, lon_max, lon_max, lon_min, lon_min],
    [lat_min, lat_min, lat_max, lat_max, lat_min],
    "k--",
    linewidth=1,
    transform=ccrs.PlateCarree(),
)

# NYC marker
ax.scatter(
    -74.0,
    40.7,
    color="black",
    s=100,
    marker="*",
    zorder=10,
    transform=ccrs.PlateCarree(),
)
ax.text(-73.5, 40.7, "NYC", fontsize=7, transform=ccrs.PlateCarree())

# Map features
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.3)
ax.add_feature(cfeature.BORDERS, linewidth=0.3)


ax.set_extent([lon_min - 5, lon_max + 5, lat_min - 5, lat_max + 5])

# Legend
legend_elements = [
    plt.Line2D(
        [0], [0], color=node_colors[(i, j)], linewidth=2, label=f"Node ({i},{j})"
    )
    for i in range(xdim)
    for j in range(ydim)
]
legend_elements += [
    plt.Line2D([0], [0], color="gray", linewidth=0.8, alpha=0.5, label="Full track"),
    plt.Line2D([0], [0], color="gray", linewidth=2.0, label=r"$\pm$48 hr window"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=6)

ax.set_title(
    "TC Tracks During Flash Flood Events\n"
    r"(full track shown; bold = $\pm$48 hr window, | = window bounds, $\bullet$ = event time)",
    fontsize=8,
)

plt.savefig(f"{fig_dir}/Z500_and_{_lbl}_som_tc_tracks.png", bbox_inches="tight")
plt.close()

print(f"Saved TC tracks map to {fig_dir}/Z500_and_{_lbl}_som_tc_tracks.png")


Saved TC tracks map to figs/Z500-and-ivt-SOM/Z500_and_ivt_som_tc_tracks.png


In [ ]:
# Compare precipitation intensity for TC vs non-TC events
# Merge TC info with precipitation data
bmu_df_with_tc = bmu_df.merge(
    tc_df[["timestamp", "tc_present", "storm_ids"]],
    on="timestamp",
    how="left"
)

print("Precipitation Statistics: TC vs Non-TC Events")
print("=" * 65)
print(f"{'Category':<20} {'N':>6} {'Mean (in)':>12} {'Median (in)':>12} {'Std (in)':>10}")
print("-" * 65)

# Overall comparison
tc_precip = bmu_df_with_tc[bmu_df_with_tc["tc_present"]]["max_precip_in"].dropna()
non_tc_precip = bmu_df_with_tc[~bmu_df_with_tc["tc_present"]]["max_precip_in"].dropna()

print(f"{'TC-Associated':<20} {len(tc_precip):>6} {tc_precip.mean():>12.2f} {tc_precip.median():>12.2f} {tc_precip.std():>10.2f}")
print(f"{'Non-TC':<20} {len(non_tc_precip):>6} {non_tc_precip.mean():>12.2f} {non_tc_precip.median():>12.2f} {non_tc_precip.std():>10.2f}")
print("-" * 65)

# Mann-Whitney U test for difference
if len(tc_precip) > 0 and len(non_tc_precip) > 0:
    stat, p = mannwhitneyu(tc_precip, non_tc_precip, alternative="two-sided")
    print(f"\nMann-Whitney U test: U={stat:.1f}, p={p:.4f}")
    if p < 0.05:
        print("→ Significant difference in precipitation intensity between TC and non-TC events")
    else:
        print("→ No significant difference in precipitation intensity")

tc_df.to_csv("data/som_2x2_tc_associations.csv", index=False)
print("Saved TC association data to data/som_2x2_tc_associations.csv")

## StageIV Gridded Precipitation by SOM Node

Per-node maximum hourly gridded precipitation extracted from the NCEP Stage IV multi-sensor QPE product (`/mnt/drive2/StageIV/stageiv_tristate_hourly.nc`). For each flash flood event the maximum 1-hour accumulation within a ±6 h window is taken over a small spatial domain covering the NYC five-borough area (40.5–40.9 °N, 74.3–73.7 °W). Precipitation is converted from kg m⁻² (mm) to inches for direct comparison with the ASOS results above.

In [37]:
import xarray as xr

STAGEIV_NC = "/mnt/drive2/StageIV/stageiv_tristate_hourly.nc"

# Lazy-load the preprocessed StageIV dataset
ds_s4 = xr.open_dataset(STAGEIV_NC, chunks={"time": 168})
s4_times = pd.DatetimeIndex(ds_s4["time"].values)

# 2-D lat/lon arrays on the HRAP grid
lat2d = ds_s4["latitude"].values   # (ny, nx)
lon2d = ds_s4["longitude"].values  # (ny, nx)

# Tight spatial mask covering the NYC five-borough area
# (~40.5–40.9 °N, 74.3–73.7 °W) — a small patch of gridboxes around the city centre
nyc_lat_min, nyc_lat_max = 40.5, 40.9
nyc_lon_min, nyc_lon_max = -74.3, -73.7

spatial_mask = (
    (lat2d >= nyc_lat_min) & (lat2d <= nyc_lat_max)
    & (lon2d >= nyc_lon_min) & (lon2d <= nyc_lon_max)
)

print(f"StageIV time range : {s4_times.min()} → {s4_times.max()}")
print(
    f"NYC domain mask    : {int(spatial_mask.sum())} grid cells "
    f"({nyc_lat_min}–{nyc_lat_max}°N, {nyc_lon_min}–{nyc_lon_max}°E)"
)

StageIV time range : 2002-05-01 00:00:00 → 2024-10-31 23:00:00
NYC domain mask    : 127 grid cells (40.5–40.9°N, -74.3–-73.7°E)


In [38]:
MM_TO_IN = 1.0 / 25.4
_window = 6  # hours (matches ASOS section)

# Identify all unique StageIV time indices needed across every event's ±window span
all_needed: set[int] = set()
event_windows: list[list[int]] = []

for _, row in bmu_df.iterrows():
    t0 = row["timestamp"]  # UTC
    time_mask = (s4_times >= t0 - pd.Timedelta(hours=_window)) & (
        s4_times <= t0 + pd.Timedelta(hours=_window)
    )
    idxs = np.where(time_mask)[0].tolist()
    event_windows.append(idxs)
    all_needed.update(idxs)

needed_idxs = sorted(all_needed)
print(f"Loading {len(needed_idxs)} unique StageIV time steps …")

# Batch-load only the needed time steps and apply fill-value mask
precip_raw = ds_s4["precipitation"].isel(time=needed_idxs).values  # (n_t, ny, nx)
precip_raw = np.where(precip_raw < 0, np.nan, precip_raw)

# Reduce to the NYC spatial mask → (n_t, n_cells)
precip_nyc = precip_raw[:, spatial_mask]

idx_to_sub = {orig: new for new, orig in enumerate(needed_idxs)}

# Compute the per-event maximum
max_precip_s4 = []
for idxs in event_windows:
    if idxs:
        sub = [idx_to_sub[i] for i in idxs]
        max_mm = np.nanmax(precip_nyc[sub, :])
        max_precip_s4.append(
            float(max_mm) * MM_TO_IN if not np.isnan(max_mm) else np.nan
        )
    else:
        max_precip_s4.append(np.nan)  # event pre-dates or outside StageIV coverage

bmu_df["max_precip_s4_in"] = max_precip_s4
n_cov = bmu_df["max_precip_s4_in"].notna().sum()
print(f"Events with StageIV coverage : {n_cov}/{len(bmu_df)}")
print(
    f"StageIV precip range         : "
    f"{bmu_df['max_precip_s4_in'].min():.2f}–{bmu_df['max_precip_s4_in'].max():.2f} in"
)


Loading 1287 unique StageIV time steps …
Events with StageIV coverage : 99/117
StageIV precip range         : 0.40–3.86 in


In [39]:
# Per-node Stage IV precipitation statistics
s4_node_stats = (
    bmu_df.dropna(subset=["max_precip_s4_in"])
    .groupby(["node_i", "node_j"])["max_precip_s4_in"]
    .agg(count="count", mean="mean", median="median", std="std")
    .round(2)
)
print(s4_node_stats)

               count  mean  median   std
node_i node_j                           
0      0          22  1.53    1.34  0.58
       1          26  1.36    1.31  0.54
1      0          30  1.30    1.26  0.43
       1          21  1.45    1.34  0.71


In [45]:
# Create histogram of max hourly Stage IV rainfall for each SOM node
fig, axes = plt.subplots(ydim, xdim, figsize=(6, 4), constrained_layout=True, dpi=600)
bins = np.arange(0, 4.01, 0.25)

for i in range(xdim):
    for j in range(ydim):
        ax = axes[j, i]

        node_data = bmu_df[(bmu_df["node_i"] == i) & (bmu_df["node_j"] == j)][
            "max_precip_s4_in"
        ].dropna()

        ax.hist(
            node_data,
            bins=bins,
            color="darkcyan",
            alpha=0.9,
            edgecolor="white",
            linewidth=0.5,
        )

        n = len(node_data)
        median = node_data.median() if n > 0 else np.nan

        if n > 0:
            ax.axvline(
                median,
                color="red",
                linestyle="--",
                linewidth=1,
                label=f'Median: {median:.2f}"',
            )

        ax.set_title(f"({i},{j})  N={n}", fontsize=6)
        ax.set_xlim(0, 4.0)
        ax.set_ylim(0, 12)
        ax.set_yticks(np.arange(0, 13, 2))
        ax.tick_params(axis="both", labelsize=5)
        ax.grid(True, linewidth=0.3, alpha=0.5, axis="y")

        if j == ydim - 1:
            ax.set_xlabel("Max Hourly Precip (in)", fontsize=5)
        if i == 0:
            ax.set_ylabel("Count", fontsize=5)

        if n > 0:
            ax.legend(fontsize=4, loc="upper right")

plt.suptitle(
    r"Maximum Hourly Stage IV Precipitation ($\pm$6 hr window) by SOM Node",
    fontsize=8,
    y=1.02,
)
plt.savefig(
    f"{fig_dir}/Z500_and_{_lbl}_som_max_precip_histograms_stageiv.png",
    bbox_inches="tight",
)
plt.close()
print(
    f"Saved Stage IV histogram to "
    f"{fig_dir}/Z500_and_{_lbl}_som_max_precip_histograms_stageiv.png"
)


Saved Stage IV histogram to figs/Z500-and-ivt-SOM/Z500_and_ivt_som_max_precip_histograms_stageiv.png


## Case Study: Hurricane Ida (1–2 September 2021)

Maximum 1-hour Stage IV precipitation over the NYC/tri-state region during the Ida event. Ida's remnants produced record-breaking flash flooding across the NYC metro on the evening of 1 September 2021 (local time).

In [43]:
from matplotlib.colors import ListedColormap, BoundaryNorm

# Hurricane Ida remnants struck NYC metro evening of 1 Sept (local) → 2 Sept UTC
ida_start = pd.Timestamp("2021-09-01 00:00:00")
ida_end   = pd.Timestamp("2021-09-02 23:59:00")

ida_mask = (s4_times >= ida_start) & (s4_times <= ida_end)
ida_idxs = np.where(ida_mask)[0].tolist()
print(
    f"Ida window: {s4_times[ida_idxs[0]]} → {s4_times[ida_idxs[-1]]} "
    f"({len(ida_idxs)} hourly steps)"
)

# Load and clean (kg m⁻² = mm; fill value < 0)
ida_precip_mm = ds_s4["precipitation"].isel(time=ida_idxs).values  # (n_t, ny, nx)
ida_precip_mm = np.where(ida_precip_mm < 0, np.nan, ida_precip_mm)

# Max hourly precip at each grid cell → inches
ida_max_in = np.nanmax(ida_precip_mm, axis=0) / 25.4

# --- NWS-style QPE colormap ---
# 10 intervals between 11 levels; values > 5 in drawn in dark red
nws_levels = [0.10, 0.25, 0.50, 0.75, 1.00, 1.50, 2.00, 2.50, 3.00, 4.00, 5.00]
nws_colors = [
    "#c8c8c8",  # 0.10–0.25 in  (light gray)
    "#04e9e7",  # 0.25–0.50     (cyan)
    "#019ff4",  # 0.50–0.75     (sky blue)
    "#0300f4",  # 0.75–1.00     (blue)
    "#02fd02",  # 1.00–1.50     (bright green)
    "#01c501",  # 1.50–2.00     (green)
    "#008e00",  # 2.00–2.50     (dark green)
    "#fdf802",  # 2.50–3.00     (yellow)
    "#fd9500",  # 3.00–4.00     (orange)
    "#fd0000",  # 4.00–5.00     (red)
]
cmap_nws = ListedColormap(nws_colors)
cmap_nws.set_over("#bc0000")  # dark red for > 5.0 in
norm_nws = BoundaryNorm(nws_levels, ncolors=len(nws_colors))

# Mask negligible values
plot_data = np.where(ida_max_in < nws_levels[0], np.nan, ida_max_in)

# --- Map ---
fig, ax = plt.subplots(
    figsize=(6, 5),
    subplot_kw={"projection": ccrs.PlateCarree()},
    dpi=300,
)

cf = ax.pcolormesh(
    lon2d, lat2d, plot_data,
    cmap=cmap_nws, norm=norm_nws,
    transform=ccrs.PlateCarree(),
    shading="auto",
)

cbar = plt.colorbar(
    cf, ax=ax, orientation="vertical", pad=0.02, shrink=0.9, extend="max"
)
cbar.set_label("Max Hourly Precip (in)", fontsize=8)
cbar.ax.tick_params(labelsize=7)

ax.add_feature(cfeature.LAND, facecolor="#f0f0f0", zorder=0)
ax.add_feature(cfeature.OCEAN, facecolor="#d0e8f0", zorder=0)
ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=2)
ax.add_feature(cfeature.STATES.with_scale("10m"), linewidth=0.4, zorder=2)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, zorder=2)

ax.scatter(
    -74.0, 40.7, color="red", s=80, marker="*", zorder=10,
    transform=ccrs.PlateCarree(),
)
ax.text(
    -73.93, 40.72, "NYC", fontsize=7, fontweight="bold",
    transform=ccrs.PlateCarree(), zorder=10,
)

# Same spatial bounds as the NYC domain mask defined above
buf = 0.05  # small buffer so gridlines aren't clipped
ax.set_extent([
    nyc_lon_min - buf, nyc_lon_max + buf,
    nyc_lat_min - buf, nyc_lat_max + buf,
])
ax.set_title(
    "Hurricane Ida: Max Hourly Stage IV Precipitation\n1–2 September 2021 (UTC)",
    fontsize=9,
)

plt.savefig(
    f"{fig_dir}/ida_stageiv_max_hourly_precip.png", bbox_inches="tight"
)
plt.close()
print(f"Saved to {fig_dir}/ida_stageiv_max_hourly_precip.png")

Ida window: 2021-09-01 00:00:00 → 2021-09-02 23:00:00 (48 hourly steps)


/tmp/ipykernel_4087711/2582038961.py:19: RuntimeWarning: All-NaN slice encountered
  ida_max_in = np.nanmax(ida_precip_mm, axis=0) / 25.4


Saved to figs/Z500-and-ivt-SOM/ida_stageiv_max_hourly_precip.png
